## Drop별 조성비 분석

하나의 drop 안에서 각 화합물이 전체의 **몇 %를 차지하는가**를 분석합니다.

### 예시
```
Drop 5 적분 결과:
  TNT     면적: 800
  RDX     면적: 200
  2,4-DNT 면적: 1000
  총합:          2000

조성비:
  TNT     = 800  / 2000 = 40%
  RDX     = 200  / 2000 = 10%
  2,4-DNT = 1000 / 2000 = 50%
```

이 비율이 **Relative Ion Abundance** (상대 이온 풍부도)입니다.

In [ ]:
!pip install -q pyopenms scipy matplotlib numpy pandas openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from scipy import integrate
import pyopenms as oms
import openpyxl, os, warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

### 데이터 로드 및 적분
mzML 데이터를 로드하고 drop별 적분을 실행합니다.

In [ ]:
# === 데이터 로드 ===
DATA_DIR = 'echo-tof-colab/data'
MZML_DIR = f'{DATA_DIR}/mzml'
TOFMS_FILE = f'{MZML_DIR}/20260330_TOFMS.mzML'
MRMHR_FILE = f'{MZML_DIR}/20260330_MRMHR_Q1Q3_Final.mzML'
EXCEL_FILE = f'{DATA_DIR}/Explosives_EchoMS_20260331.xlsx'

def load_mzml(filepath):
    """mzML 파일을 로드하여 MSExperiment 객체 반환"""
    exp = oms.MSExperiment()
    oms.MzMLFile().load(filepath, exp)
    print(f'로드 완료: {os.path.basename(filepath)} ({exp.getNrSpectra()} spectra)')
    return exp

exp_tof = load_mzml(TOFMS_FILE)

def extract_tic(exp, ms_level=1):
    """TIC 추출"""
    rt_list, tic_list = [], []
    for spec in exp:
        if spec.getMSLevel() == ms_level:
            rt_list.append(spec.getRT() / 60.0)
            mz, intensity = spec.get_peaks()
            tic_list.append(np.sum(intensity))
    return np.array(rt_list), np.array(tic_list)

rt_tof, tic_tof = extract_tic(exp_tof)

def extract_xic(exp, target_mz, tolerance_ppm=20.0, ms_level=1):
    """XIC 추출: target_mz ± tolerance 범위의 intensity 합"""
    rt_list, int_list = [], []
    tol = target_mz * tolerance_ppm * 1e-6
    for spec in exp:
        if spec.getMSLevel() == ms_level:
            rt_list.append(spec.getRT() / 60.0)
            mz, ints = spec.get_peaks()
            mask = (mz >= target_mz - tol) & (mz <= target_mz + tol)
            int_list.append(float(ints[mask].sum()) if mask.any() else 0.0)
    return np.array(rt_list), np.array(int_list)

# 화합물 스킴 로드
wb = openpyxl.load_workbook(EXCEL_FILE, data_only=True)
ws_std = wb['STD list']
compounds = []
for row in ws_std.iter_rows(min_row=2, max_row=16, values_only=True):
    if row[0] and row[1]:
        compounds.append({'name': str(row[2] or row[1]), 'full_name': str(row[1]), 'mz': None})

# XIC 추출
# Target compounds — multi-adduct + isotope 통합
# Total Area = Σ_Adducts( Σ_Isotopes( Area(Adduct_i, Isotope_j) ) )

ISOTOPE_OFFSETS = [0.0, 1.00335, 2.00671]  # M, M+1, M+2
ISOTOPE_LABELS = ['M', 'M+1', 'M+2']

target_compounds = {
    'TNT': {
        'MW': 227.018,
        'adducts': [
            {'name': '[M-H]-',    'mz': 226.009},
            {'name': '[M+Cl]-',   'mz': 261.986},
        ]
    },
    'RDX': {
        'MW': 222.039,
        'adducts': [
            {'name': '[M+CH3COO]-', 'mz': 281.048},
            {'name': '[M+Cl]-',   'mz': 256.000},
        ]
    },
    'HMX': {
        'MW': 296.047,
        'adducts': [
            {'name': '[M+CH3COO]-', 'mz': 355.0592},
            {'name': '[M+Cl]-',   'mz': 330.008},
        ]
    },
    '2,4-DNT': {
        'MW': 182.019,
        'adducts': [
            {'name': '[M-H]-',    'mz': 181.0124},
        ]
    },
    'Tetryl': {
        'MW': 287.025,
        'adducts': [
            {'name': '[M-NO2]-',   'mz': 241.0204},
        ]
    },
    '4-Am-DNT': {
        'MW': 197.046,
        'adducts': [
            {'name': '[M-H]-',    'mz': 196.0358},
        ]
    },
    '2,6-DNT': {
        'MW': 182.019,
        'adducts': [
            {'name': '[M]-',      'mz': 182.018},
        ]
    },
}


def extract_xic_full(exp, adducts, tolerance_ppm=20.0, ms_level=1):
    """Isotope + Adduct 통합 XIC 추출

    각 adduct의 M, M+1, M+2 동위원소 피크를 개별 추출 후 전부 합산.
    Total XIC = Σ_Adducts( Σ_Isotopes( XIC(adduct_i, isotope_j) ) )
    """
    rts = None
    total_xic = None
    detail = {}  # {adduct_name: {isotope_label: xic_array}}

    for adduct in adducts:
        base_mz = adduct['mz']
        adduct_name = adduct['name']
        detail[adduct_name] = {}

        for iso_offset, iso_label in zip(ISOTOPE_OFFSETS, ISOTOPE_LABELS):
            target_mz = base_mz + iso_offset
            tol = target_mz * tolerance_ppm * 1e-6
            rt_list, int_list = [], []

            for spec in exp:
                if spec.getMSLevel() == ms_level:
                    if rts is None:
                        rt_list.append(spec.getRT() / 60.0)
                    mzs, ints = spec.get_peaks()
                    mask = (mzs >= target_mz - tol) & (mzs <= target_mz + tol)
                    int_list.append(float(ints[mask].sum()) if mask.any() else 0.0)

            xic = np.array(int_list)
            if rts is None:
                rts = np.array(rt_list)
            detail[adduct_name][iso_label] = xic

            if total_xic is None:
                total_xic = xic.copy()
            else:
                total_xic += xic

    return rts, total_xic, detail


# XIC 추출 (Isotope + Adduct 통합)
xic_data = {}
for name, info in target_compounds.items():
    rt, total, detail = extract_xic_full(exp_tof, info['adducts'])
    xic_data[name] = {
        'rt': rt, 'intensity': total, 'detail': detail,
        'adducts': info['adducts'],
    }

    # Report
    total_max = total.max()
    print(f'{name:10s} | Total max: {total_max:.1f}')
    for aname, isotopes in detail.items():
        for ilabel, ixic in isotopes.items():
            imax = ixic.max()
            if imax > 0:
                pct = imax / total_max * 100 if total_max > 0 else 0
                print(f'  {aname:15s} {ilabel:4s}: max={imax:10.1f} ({pct:5.1f}%)')

detected = {k: v for k, v in xic_data.items() if v['intensity'].max() > 0}
print(f"\nDetected: {len(detected)}/{len(target_compounds)}")

# Target compounds — multi-adduct + isotope 통합
# Total Area = Σ_Adducts( Σ_Isotopes( Area(Adduct_i, Isotope_j) ) )

ISOTOPE_OFFSETS = [0.0, 1.00335, 2.00671]  # M, M+1, M+2
ISOTOPE_LABELS = ['M', 'M+1', 'M+2']

target_compounds = {
    'TNT': {
        'MW': 227.018,
        'adducts': [
            {'name': '[M-H]-',    'mz': 226.009},
            {'name': '[M+Cl]-',   'mz': 261.986},
        ]
    },
    'RDX': {
        'MW': 222.039,
        'adducts': [
            {'name': '[M+CH3COO]-', 'mz': 281.048},
            {'name': '[M+Cl]-',   'mz': 256.000},
        ]
    },
    'HMX': {
        'MW': 296.047,
        'adducts': [
            {'name': '[M+CH3COO]-', 'mz': 355.0592},
            {'name': '[M+Cl]-',   'mz': 330.008},
        ]
    },
    '2,4-DNT': {
        'MW': 182.019,
        'adducts': [
            {'name': '[M-H]-',    'mz': 181.0124},
        ]
    },
    'Tetryl': {
        'MW': 287.025,
        'adducts': [
            {'name': '[M-NO2]-',   'mz': 241.0204},
        ]
    },
    '4-Am-DNT': {
        'MW': 197.046,
        'adducts': [
            {'name': '[M-H]-',    'mz': 196.0358},
        ]
    },
    '2,6-DNT': {
        'MW': 182.019,
        'adducts': [
            {'name': '[M]-',      'mz': 182.018},
        ]
    },
}


def extract_xic_full(exp, adducts, tolerance_ppm=20.0, ms_level=1):
    """Isotope + Adduct 통합 XIC 추출

    각 adduct의 M, M+1, M+2 동위원소 피크를 개별 추출 후 전부 합산.
    Total XIC = Σ_Adducts( Σ_Isotopes( XIC(adduct_i, isotope_j) ) )
    """
    rts = None
    total_xic = None
    detail = {}  # {adduct_name: {isotope_label: xic_array}}

    for adduct in adducts:
        base_mz = adduct['mz']
        adduct_name = adduct['name']
        detail[adduct_name] = {}

        for iso_offset, iso_label in zip(ISOTOPE_OFFSETS, ISOTOPE_LABELS):
            target_mz = base_mz + iso_offset
            tol = target_mz * tolerance_ppm * 1e-6
            rt_list, int_list = [], []

            for spec in exp:
                if spec.getMSLevel() == ms_level:
                    if rts is None:
                        rt_list.append(spec.getRT() / 60.0)
                    mzs, ints = spec.get_peaks()
                    mask = (mzs >= target_mz - tol) & (mzs <= target_mz + tol)
                    int_list.append(float(ints[mask].sum()) if mask.any() else 0.0)

            xic = np.array(int_list)
            if rts is None:
                rts = np.array(rt_list)
            detail[adduct_name][iso_label] = xic

            if total_xic is None:
                total_xic = xic.copy()
            else:
                total_xic += xic

    return rts, total_xic, detail


# XIC 추출 (Isotope + Adduct 통합)
xic_data = {}
for name, info in target_compounds.items():
    rt, total, detail = extract_xic_full(exp_tof, info['adducts'])
    xic_data[name] = {
        'rt': rt, 'intensity': total, 'detail': detail,
        'adducts': info['adducts'],
    }

    # Report
    total_max = total.max()
    print(f'{name:10s} | Total max: {total_max:.1f}')
    for aname, isotopes in detail.items():
        for ilabel, ixic in isotopes.items():
            imax = ixic.max()
            if imax > 0:
                pct = imax / total_max * 100 if total_max > 0 else 0
                print(f'  {aname:15s} {ilabel:4s}: max={imax:10.1f} ({pct:5.1f}%)')

detected = {k: v for k, v in xic_data.items() if v['intensity'].max() > 0}
print(f"\nDetected: {len(detected)}/{len(target_compounds)}")


### Step 1. Drop별 적분 데이터 준비

05장의 `integration_matrix`를 사용합니다.
독립 실행 시에는 아래 셀에서 적분까지 자동 실행됩니다.

In [ ]:
from scipy.signal import find_peaks
from scipy import integrate

def collect_all_spikes(exp, compounds, tolerance_ppm=20.0, height_factor=20.0, distance=3):
    """모든 화합물의 Isotope+Adduct 통합 XIC에서 spike 수집"""
    rts = [spec.getRT() for spec in exp if spec.getMSLevel() == 1]
    rts = np.array(rts)

    all_spike_rts = []
    compound_spikes = {}

    for name, info in compounds.items():
        # Isotope + Adduct 통합 XIC
        adducts = info.get('adducts', [{'mz': info.get('mz', 0)}])
        xic = np.zeros(len(rts))

        scan_idx = 0
        for spec in exp:
            if spec.getMSLevel() == 1:
                mzs, ints = spec.get_peaks()
                for adduct in adducts:
                    base_mz = adduct['mz']
                    for iso_offset in ISOTOPE_OFFSETS:
                        target_mz = base_mz + iso_offset
                        tol = target_mz * tolerance_ppm * 1e-6
                        mask = (mzs >= target_mz - tol) & (mzs <= target_mz + tol)
                        if mask.any():
                            xic[scan_idx] += float(ints[mask].sum())
                scan_idx += 1

        nonzero = xic[xic > 0]
        if len(nonzero) == 0:
            compound_spikes[name] = {'rt': rts, 'xic': xic, 'peaks': np.array([], dtype=int)}
            continue

        threshold = np.median(nonzero) * height_factor
        peaks, _ = find_peaks(xic, height=threshold, distance=distance)
        compound_spikes[name] = {'rt': rts, 'xic': xic, 'peaks': peaks}
        all_spike_rts.extend(rts[peaks])

    return np.array(sorted(all_spike_rts)), compound_spikes, rts


def cluster_drop_events(spike_rts, merge_window=1.0):
    """Cluster nearby spikes into drop events"""
    if len(spike_rts) == 0:
        return []
    clusters, current = [], [spike_rts[0]]
    for rt in spike_rts[1:]:
        if rt - current[-1] < merge_window:
            current.append(rt)
        else:
            clusters.append(current)
            current = [rt]
    clusters.append(current)
    return [{'drop_id': i+1, 'center_rt': np.mean(cl), 'n_compounds': len(cl)}
            for i, cl in enumerate(clusters)]


def integrate_per_drop(scan_rts, xic, drop_table, half_window=0.921):
    """Integrate XIC within each drop window (Trapezoidal Rule)"""
    results = []
    for drop in drop_table:
        center = drop['center_rt']
        mask = (scan_rts >= center - half_window) & (scan_rts <= center + half_window)
        rt_win, int_win = scan_rts[mask], xic[mask]
        if len(rt_win) < 2:
            results.append({'drop_id': drop['drop_id'], 'area': 0.0, 'height': 0.0})
            continue
        corrected = np.clip(int_win, 0, None)
        results.append({'drop_id': drop['drop_id'],
                        'area': integrate.trapezoid(corrected, rt_win),
                        'height': corrected.max()})
    return results


### Step 2. Drop별 조성비 계산

각 drop 안에서 화합물별 면적의 **비율(%)**을 계산합니다.

$$\text{조성비}(\text{TNT}) = \frac{\text{Area}(\text{TNT})}{\sum \text{Area}(\text{모든 화합물})} \times 100\%$$

In [ ]:
# --- Drop별 조성비 계산 ---
compound_names = list(integration_matrix.keys())
n_drops = len(drop_table)
n_comp = len(compound_names)

# Area matrix: (compounds x drops)
area_matrix = np.zeros((n_comp, n_drops))
for ci, name in enumerate(compound_names):
    for di, r in enumerate(integration_matrix[name]):
        area_matrix[ci, di] = r['area']

# Composition matrix: (compounds x drops) — percent within each drop
comp_matrix = np.zeros_like(area_matrix)
for di in range(n_drops):
    total = area_matrix[:, di].sum()
    if total > 0:
        comp_matrix[:, di] = area_matrix[:, di] / total * 100

# Per-drop results table
print(f'{"Drop":>5s} {"RT(s)":>8s} ', end='')
for name in compound_names:
    print(f'{name:>10s}', end='')
print(f'{"  Total Area":>14s}')
print('-' * (15 + 10 * n_comp + 14))

valid_drops = []  # drops where at least 2 compounds detected
for di in range(n_drops):
    total = area_matrix[:, di].sum()
    n_detected = np.sum(area_matrix[:, di] > 0)
    rt = drop_table[di]['center_rt']
    
    print(f'{di+1:5d} {rt:8.1f} ', end='')
    for ci in range(n_comp):
        if comp_matrix[ci, di] > 0:
            print(f'{comp_matrix[ci, di]:9.1f}%', end='')
        else:
            print(f'{"—":>10s}', end='')
    print(f'{total:14.2e}')
    
    if n_detected >= 2:
        valid_drops.append(di)

print(f'\nValid drops (>= 2 compounds): {len(valid_drops)}/{n_drops}')


### Step 3. 최종 조성비

여러 drop의 조성비를 종합합니다.

In [ ]:
# --- 유효 drop에서 평균 조성비 ---
valid_comp = comp_matrix[:, valid_drops]  # (compounds x valid_drops)

print(f'{"Compound":12s} {"Mean %":>8s} {"SD":>8s} {"RSD%":>8s} {"Min %":>8s} {"Max %":>8s} {"N drops":>8s}')
print('-' * 65)

final_composition = {}
for ci, name in enumerate(compound_names):
    vals = valid_comp[ci, :]
    nonzero = vals[vals > 0]
    if len(nonzero) > 0:
        mean_pct = np.mean(nonzero)
        sd = np.std(nonzero) if len(nonzero) > 1 else 0
        rsd = sd / mean_pct * 100 if mean_pct > 0 else 0
        final_composition[name] = {'mean': mean_pct, 'sd': sd, 'rsd': rsd, 'n': len(nonzero)}
        print(f'{name:12s} {mean_pct:8.1f} {sd:8.1f} {rsd:7.1f}% {nonzero.min():8.1f} {nonzero.max():8.1f} {len(nonzero):8d}')
    else:
        final_composition[name] = {'mean': 0, 'sd': 0, 'rsd': 0, 'n': 0}
        print(f'{name:12s} {"not detected":>42s}')


### 시각화

- **Stacked Bar**: 각 drop에서 화합물별 조성비
- **Pie Chart**: 전체 평균 조성비

In [1]:
# --- Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Stacked bar: drop별 조성비
ax = axes[0, 0]
bottom = np.zeros(len(valid_drops))
drop_labels = [f'D{valid_drops[i]+1}' for i in range(len(valid_drops))]
for ci, name in enumerate(compound_names):
    vals = valid_comp[ci, :]
    ax.bar(range(len(valid_drops)), vals, bottom=bottom, label=name)
    bottom += vals
ax.set_xticks(range(len(valid_drops)))
ax.set_xticklabels(drop_labels, rotation=45, fontsize=7)
ax.set_ylabel('Composition (%)')
ax.set_title('Per-Drop Composition (Stacked)')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(0, 105)

# 2. Pie chart: mean composition
ax = axes[0, 1]
detected = {k: v for k, v in final_composition.items() if v['mean'] > 0}
labels = list(detected.keys())
sizes = [detected[k]['mean'] for k in labels]
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
ax.set_title('Mean Composition')

# 3. Box plot: drop간 조성비 분포
ax = axes[1, 0]
box_data = []
box_labels = []
for ci, name in enumerate(compound_names):
    vals = valid_comp[ci, :]
    nonzero = vals[vals > 0]
    if len(nonzero) > 0:
        box_data.append(nonzero)
        box_labels.append(name)
ax.boxplot(box_data, labels=box_labels)
ax.set_ylabel('Composition (%)')
ax.set_title('Composition Distribution Across Drops')
ax.grid(alpha=0.3)

# 4. RSD bar chart
ax = axes[1, 1]
rsd_names = [k for k in compound_names if final_composition[k]['n'] > 1]
rsd_vals = [final_composition[k]['rsd'] for k in rsd_names]
colors = ['green' if r < 20 else 'orange' if r < 50 else 'red' for r in rsd_vals]
ax.barh(rsd_names, rsd_vals, color=colors)
ax.axvline(20, color='green', linestyle='--', alpha=0.5, label='20% threshold')
ax.set_xlabel('RSD (%)')
ax.set_title('Reproducibility (Composition RSD%)')
ax.legend()

plt.tight_layout()
plt.show()


### Summary

| 단계 | 내용 |
|------|------|
| Drop 검출 | XIC spike 클러스터링 → 공통 drop 타이밍 테이블 |
| Drop별 적분 | 각 drop 윈도우 내 Trapezoidal Rule 적분 |
| **Drop별 조성비** | **drop 내 화합물 면적비(%) — 하나의 실험 결과** |
| 최종 조성비 | 유효 drop들의 평균 ± RSD% |

**1 drop = 1 실험.** Drop 간 면적을 합산하지 않고, 각 drop의 조성비를 독립적으로 계산합니다.

---

⬅️ **Prev:** [Parameter Sensitivity](07%20Parameter%20Sensitivity.ipynb) | **Next:** [m/z Prediction](09%20mz%20Prediction.ipynb) ➡️